# Step 2 – Dynamic Data Test Guide

Notebook này giúp các thành viên trong nhóm test nhanh **Bước 2 – Tạo dữ liệu động**.

Quy trình:
1. Kiểm tra cấu trúc `src/data_generation/`.
2. Chuẩn hóa giá về VND.
3. Kiểm tra hoặc tạo `review_bank.json`.
4. Sinh `dynamic_transactions.csv`.
5. Chạy validation cuối.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

current = Path.cwd().resolve()
ROOT = None

for p in [current, *current.parents]:
    if (p / "README.md").exists() and (p / "data").exists() and (p / "src").exists():
        ROOT = p
        break

assert ROOT is not None, "Không tìm thấy thư mục gốc repo."
print("Repo root:", ROOT)
print("Python:", sys.executable)


Repo root: C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11
Python: c:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\.venv\Scripts\python.exe


## 1. Kiểm tra cấu trúc script

Ba script Step 2 phải nằm trong `src/data_generation/`.


In [2]:
SCRIPT_DIR = ROOT / "src" / "data_generation"

scripts = {
    "standardize_currency": SCRIPT_DIR / "standardize_currency.py",
    "generate_review_bank": SCRIPT_DIR / "generate_review_bank.py",
    "generate_dynamic_data": SCRIPT_DIR / "generate_dynamic_data.py",
}

for name, path in scripts.items():
    print(f"{name:25} ->", "OK" if path.exists() else "MISSING", path)
    assert path.exists(), f"Thiếu file: {path}"

print("Script structure: PASSED")


standardize_currency      -> OK C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11\src\data_generation\standardize_currency.py
generate_review_bank      -> OK C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11\src\data_generation\generate_review_bank.py
generate_dynamic_data     -> OK C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11\src\data_generation\generate_dynamic_data.py
Script structure: PASSED


## 2. Chuẩn hóa tiền tệ về VND


In [3]:
subprocess.run(
    [sys.executable, str(scripts["standardize_currency"])],
    cwd=ROOT,
    check=True
)

tesco_path = ROOT / "data" / "processed" / "tesco_products_vnd.csv"
tiki_path = ROOT / "data" / "processed" / "tiki_products_vnd.csv"

tesco = pd.read_csv(tesco_path)
tiki = pd.read_csv(tiki_path)

assert tesco["Source_Currency"].eq("GBP").all()
assert tiki["Source_Currency"].eq("VND").all()
assert tiki["Discount_Price_VND"].notna().all()

print("Tesco rows:", len(tesco))
print("Tiki rows:", len(tiki))
print("Tesco missing Discount_Price_VND:", tesco["Discount_Price_VND"].isna().sum())
print("Currency validation: PASSED")


Tesco rows: 1200
Tiki rows: 5359
Tesco missing Discount_Price_VND: 3
Currency validation: PASSED


## 3. Review Bank

Mặc định **không tạo lại** review bank vì Ollama tốn thời gian.
Đổi `REGENERATE_REVIEW_BANK = True` nếu muốn test lại Ollama.


In [4]:
REGENERATE_REVIEW_BANK = False
review_bank_path = ROOT / "data" / "processed" / "review_bank.json"

if REGENERATE_REVIEW_BANK:
    ollama_cmd = shutil.which("ollama")

    if ollama_cmd is None and os.name == "nt":
        local_appdata = os.environ.get("LOCALAPPDATA")
        if local_appdata:
            fallback = Path(local_appdata) / "Programs" / "Ollama" / "ollama.exe"
            if fallback.exists():
                ollama_cmd = str(fallback)

    assert ollama_cmd is not None, "Không tìm thấy Ollama."

    result = subprocess.run(
        [ollama_cmd, "list"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
    assert "qwen3:1.7b" in result.stdout, "Chưa có model qwen3:1.7b."

    subprocess.run(
        [sys.executable, str(scripts["generate_review_bank"])],
        cwd=ROOT,
        check=True
    )

assert review_bank_path.exists(), "Thiếu review_bank.json."

with open(review_bank_path, "r", encoding="utf-8") as f:
    review_bank = json.load(f)

for rating in range(1, 6):
    key = str(rating)
    assert key in review_bank
    assert len(review_bank[key]) == 20

assert sum(len(v) for v in review_bank.values()) == 100

print("Review bank validation: PASSED")
print("Total reviews:", sum(len(v) for v in review_bank.values()))


Review bank validation: PASSED
Total reviews: 100


## 4. Sinh Dynamic Data


In [5]:
subprocess.run(
    [sys.executable, str(scripts["generate_dynamic_data"])],
    cwd=ROOT,
    check=True
)

dynamic_path = ROOT / "data" / "processed" / "dynamic_transactions.csv"
assert dynamic_path.exists(), "Không tạo được dynamic_transactions.csv."

print("Dynamic data generated:", dynamic_path)


Dynamic data generated: C:\Users\LucasNguyen\Desktop\NMPTDL&AI\Prac_T02\Prac_T02\data-analysis-ai-nhom11\data\processed\dynamic_transactions.csv


## 5. Validation cuối Step 2


In [6]:
df = pd.read_csv(dynamic_path)

expected_columns = [
    "Transaction_ID",
    "Customer_ID",
    "Product_ID",
    "Transaction_Date",
    "Quantity",
    "Unit_Price_VND",
    "Revenue",
    "Rating",
    "Customer_Review",
    "Gender",
    "Age",
    "City",
]

assert df.shape == (10000, 12), f"Shape sai: {df.shape}"
assert list(df.columns) == expected_columns
assert df["Transaction_ID"].nunique() == 10000
assert df.isna().sum().sum() == 0
assert df["Quantity"].between(1, 5).all()
assert df["Rating"].between(1, 5).all()
assert (df["Revenue"] == df["Unit_Price_VND"] * df["Quantity"]).all()

valid_products = pd.concat([tesco, tiki], ignore_index=True)
valid_products = valid_products.dropna(subset=["Discount_Price_VND"])
assert df["Product_ID"].isin(valid_products["Product_ID"]).all()

for rating in range(1, 6):
    rows = df[df["Rating"] == rating]
    valid_reviews = set(review_bank[str(rating)])
    assert rows["Customer_Review"].isin(valid_reviews).all()

print("STEP 2 VALIDATION PASSED")
print("Transactions :", len(df))
print("Customers    :", df["Customer_ID"].nunique())
print("Products used:", df["Product_ID"].nunique())
print("\nRating distribution:")
display(df["Rating"].value_counts().sort_index())
display(df.head())


STEP 2 VALIDATION PASSED
Transactions : 10000
Customers    : 1988
Products used: 5102

Rating distribution:


Rating
1    1994
2    2024
3    1932
4    1976
5    2074
Name: count, dtype: int64

,Transaction_ID,Customer_ID,Product_ID,Transaction_Date,Quantity,Unit_Price_VND,Revenue,Rating,Customer_Review,Gender,Age,City
0,T000001,C01448,53111744,2026-05-24 07:53:23,2,119000,238000,1,Tôi cảm thấy rất thất vọng vì sản phẩm không đ...,Female,39,Huế
1,T000002,C01236,10159834,2025-10-31 05:40:50,3,38458,115374,3,"Trải nghiệm không quá tốt, và không có gì nổi ...",Male,47,Vũng Tàu
2,T000003,C00811,10169434,2025-09-27 17:07:01,1,593100,593100,4,Tôi có trải nghiệm tích cực và nhìn chung hài ...,Female,63,Đà Nẵng
3,T000004,C00809,174036467,2026-07-02 23:45:05,3,359000,1077000,5,Trải nghiệm với sản phẩm rất tích cực và khiến...,Female,56,Đà Lạt
4,T000005,C01558,45468604,2026-01-02 19:27:32,1,503000,503000,4,"Với trải nghiệm này, tôi có trải nghiệm tích c...",Female,50,Hải Phòng


## Kết quả mong đợi

Nếu tất cả cell chạy không lỗi và xuất hiện `STEP 2 VALIDATION PASSED` thì Step 2 hoạt động đúng.

Các file đầu ra chính:
- `data/processed/dynamic_transactions.csv`
- `data/processed/review_bank.json`
- `data/processed/tesco_products_vnd.csv`
- `data/processed/tiki_products_vnd.csv`
